# Internship Task 3: Conditional Colorization 



In [ ]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
os.environ['CUDA_MODULE_LOADING'] = 'LAZY'

import torch
import numpy as np
from PIL import Image
import cv2
import gc
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
import gradio as gr

def clear_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"--- Task 3 Optimization Engine ---")
print(f"Device: {device} | Precision: {dtype}")

A matching Triton is not available, some optimizations will not be enabled.
Error caught was: No module named 'triton'


--- Task 3 Optimization Engine ---
Device: cuda | Precision: torch.float16


In [ ]:
clear_memory()
print("Loading Optimized Model Stack (4GB VRAM Profile)...")
controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_canny", torch_dtype=dtype)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet, torch_dtype=dtype
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

if device == "cuda":

    pipe.enable_model_cpu_offload() 
    pipe.enable_vae_slicing()
    pipe.enable_vae_tiling()
    try:
        pipe.enable_xformers_memory_efficient_attention()
        print("Optimization: XFORMERS ENABLED")
    except:
        pipe.enable_attention_slicing("max")
        print("Optimization: ATTENTION SLICING ENABLED")
else:
    pipe.to(device)

print("Conditional Colorization Engine READY!")

Loading Optimized Model Stack (4GB VRAM Profile)...


d:\Downloads\image_gen_colorization\.venv\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
safety_checker\model.safetensors not found


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.


Optimization: XFORMERS ENABLED
Conditional Colorization Engine READY!


In [ ]:
def conditional_colorization(image, condition, content_desc=""):
    clear_memory()

    image = image.resize((512, 512))

    img_np = np.array(image)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    canny = cv2.Canny(gray, 100, 200)
    canny = np.stack([canny, canny, canny], axis=2)
    canny_map = Image.fromarray(canny)
    
    styles = {
        "Golden Hour": "warm sunlight, sunset glow, orange and yellow light, soft shadows",
        "Cyberpunk": "neon city lights, blue and pink glow, futuristic night, vibrant",
        "Overcast": "soft diffused light, foggy, gray and cold tones, cinematic",
        "Vivid Day": "bright direct sunlight, clear sky, highly saturated vibrant colors",
        "Teal & Orange": "cinematic movie look, teal shadows, orange highlights"
    }
    
    style_info = styles.get(condition, "realistic photography colors")
    prompt = f"{content_desc}, {style_info}, masterpiece, 8k, highly detailed"
    negative = "blur, low quality, black and white, monochrome, artifact, unrealistic"
    
    print(f"Generating with style: {condition}...")
    result = pipe(
        prompt, 
        image=canny_map, 
        negative_prompt=negative,
        num_inference_steps=8,  
        guidance_scale=7.5
    ).images[0]
    
    return result

In [ ]:
with gr.Blocks(title="Task 3 Optimized") as demo:
    gr.Markdown("# Task 3:Conditional Colorization")
    
    with gr.Row():
        with gr.Column():
            input_img = gr.Image(label="Source Image", type="pil")
            cond = gr.Radio(
                choices=["Golden Hour", "Cyberpunk", "Overcast", "Vivid Day", "Teal & Orange"], 
                value="Golden Hour", 
                label="Atmospheric Condition"
            )
            desc = gr.Textbox(label="Image Description", placeholder="e.g. A mountain landscape")
            btn = gr.Button("Generate Colors", variant="primary")
        
        with gr.Column():
            output_img = gr.Image(label="Result")
    
    btn.click(fn=conditional_colorization, inputs=[input_img, cond, desc], outputs=output_img)

print("Launch initialized on local and public tunnels...")
demo.queue().launch(share=True)

Launch initialized on local and public tunnels...
Running on local URL:  http://127.0.0.1:7860
IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://291448bee3af006659.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Generating with style: Golden Hour...


  0%|          | 0/8 [00:00<?, ?it/s]

Generating with style: Golden Hour...


  0%|          | 0/8 [00:00<?, ?it/s]

Generating with style: Vivid Day...


  0%|          | 0/8 [00:00<?, ?it/s]

: 